Day 15 — Filter Engine Core

• Write src/screener/engine.py — load screener_config.yaml and apply threshold filters to financial_ratios

DataFrame

• Support all 15 filterable metrics: ROE min, D/E max, FCF min, Revenue CAGR 5yr min, PAT CAGR 5yr min,

OPM min, P/E max, P/B max, Dividend Yield min, ICR min, Market Cap min, Net Profit min, EPS CAGR min, Asset

Turnover min, Sales min

• D/E filter: automatically skip companies in Financials sector when D/E max filter is applied

• ICR filter: treat Debt Free label as ICR = infinity (always passes any ICR minimum threshold)

• Return sorted DataFrame with composite_quality_score column added


In [ ]:
# tests/screener/test_engine_day15_16.py
import sqlite3
import numpy as np
import pandas as pd
import pytest
import sys
sys.path.insert(0, r"C:\Users\bhrra\Desktop\nifty100")
from engine import *


def test_debt_free_icr_becomes_infinity():
    df = pd.DataFrame({
        "interest_coverage": [5.0, np.nan],
        "icr_label": ["", "Debt Free"],
    })
    out = preprocess_special_metrics(df)
    assert out["ICR_effective"].iloc[1] == np.inf
    assert out["ICR_effective"].iloc[0] == 5.0


def test_strict_operator_not_inclusive():
    """ROE exactly 15 must FAIL a '>' 15 filter (strict, not >=)."""
    df = pd.DataFrame({
        "company_id": [1, 2],
        "return_on_equity_pct": [15.0, 15.01],
        "debt_to_equity": [0.5, 0.5],
        "broad_sector": ["IT", "IT"],
        "interest_coverage": [10, 10],
        "icr_label": ["", ""],
    })
    out = apply_custom_filters(df, [{"metric": "return_on_equity_pct", "operator": ">", "value": 15}])
    assert len(out) == 1
    assert out.iloc[0]["company_id"] == 2  # only the strictly-greater row passes


def test_financials_sector_de_exempt():
    df = pd.DataFrame({
        "company_id": [1, 2],
        "broad_sector": ["Financials", "Information Technology"],
        "debt_to_equity": [8.0, 8.0],
        "return_on_equity_pct": [20, 20],
        "interest_coverage": [5, 5],
        "icr_label": ["", ""],
    })
    out = apply_custom_filters(df, [
        {"metric": "debt_to_equity", "operator": "<", "value": 1.0},
        {"metric": "return_on_equity_pct", "operator": ">", "value": 15},
    ])
    assert len(out) == 1
    assert out.iloc[0]["company_id"] == 1  # Financials passes despite D/E=8.0


def test_debt_free_blue_chip_float_equality():
    df = pd.DataFrame({
        "company_id": [1, 2],
        "broad_sector": ["FMCG", "FMCG"],
        "debt_to_equity": [0.0000001, 0.05],  # first is float-noise-zero
        "return_on_equity_pct": [20, 20],
        "sales": [6000, 6000],
        "interest_coverage": [10, 10],
        "icr_label": ["", ""],
    })
    out = apply_custom_filters(df, [
        {"metric": "debt_to_equity", "operator": "==", "value": 0},
        {"metric": "return_on_equity_pct", "operator": ">", "value": 12},
        {"metric": "sales", "operator": ">", "value": 5000},
    ])
    assert len(out) == 1
    assert out.iloc[0]["company_id"] == 1

Day 16 — 6 Preset Screeners

• Implement Quality Compounder preset: ROE > 15%, D/E < 1.0, FCF > 0, Revenue CAGR 5yr > 10%

• Implement Value Pick preset: P/E < 20, P/B < 3.0, D/E < 2.0, Dividend Yield > 1%

• Implement Growth Accelerator preset: PAT CAGR 5yr > 20%, Revenue CAGR 5yr > 15%, D/E < 2.0

• Implement Dividend Champion preset: Dividend Yield > 2%, Dividend Payout < 80%, FCF > 0

• Implement Debt-Free Blue Chip preset: D/E = 0, ROE > 12%, Revenue > 5000 Crore

• Implement Turnaround Watch preset: Revenue CAGR 3yr > 10%, FCF positive in latest year, D/E declining

year-over-year

• Test each preset on full 92-company universe — verify each returns between 5 and 50 companies and results

make business sense



In [ ]:
# Cell 1 — imports
import sys
from pathlib import Path
project_root = Path(r"C:\Users\bhrra\Desktop\nifty100")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import sqlite3
from src.screener.engine import (
    build_screener_universe, run_all_presets, validate_preset_counts, load_screener_config,
)
print("Imports OK")

In [ ]:
# Cell 2 — connection
conn = sqlite3.connect(r"C:\Users\bhrra\Desktop\nifty100\db\nifty100.db")  # apna exact path
print("Connected:", conn)

In [ ]:
# Cell 3 — universe build (yeh sabse likely fail point hai)
universe = build_screener_universe(conn)
print("Universe shape:", universe.shape)
print(universe.head())

In [ ]:
# # Notebook ke sabse pehle cell mein
# import sys
# from pathlib import Path

# project_root = Path(r"C:\Users\bhrra\Desktop\nifty100")  # apna actual root path
# if str(project_root) not in sys.path:
#     sys.path.insert(0, str(project_root))

# # Ab yeh chalega
# from src.screener.engine import (
#     apply_custom_filters, preprocess_special_metrics,
#     compute_turnaround_watch, run_all_presets, validate_preset_counts,
#     load_screener_config, build_screener_universe,
# )

In [ ]:
# Cell 4 — config load
config = load_screener_config(r"C:\Users\bhrra\Desktop\nifty100\src\screener\screener_config.yaml")
print("Config loaded:", list(config["presets"].keys()))

In [ ]:
# Cell 5 — run presets
results = run_all_presets(universe, conn, config)
print("Presets run:", list(results.keys()))

In [ ]:
# Cell 6 — summary
summary = validate_preset_counts(results)
print(summary)

In [ ]:
import os
db_path = r"C:\Users\bhrra\Desktop\nifty100\db\nifty100.db"
print("File exists:", os.path.exists(db_path))
print("File size (bytes):", os.path.getsize(db_path) if os.path.exists(db_path) else "N/A")

In [ ]:
import sys
import engine

print(engine.__file__)
print(hasattr(engine, "apply_custom_filters"))
print([x for x in dir(engine) if "filter" in x.lower()])

Day 17 — Composite Score & Export

• Implement composite quality score (0 to 100 scale): 35% Profitability (ROE 15% + ROCE 10% + NPM 10%) +

30% Cash Quality (FCF CAGR 15% + CFO/PAT ratio 10% + FCF positive flag 5%) + 20% Growth (Revenue

CAGR 10% + PAT CAGR 10%) + 15% Leverage (D/E score 10% + ICR score 5%)

• Normalise each metric using P10/P90 winsorisation — cap extreme values at 10th and 90th percentile before

scaling to 0-100

• Compute sector-relative composite score — normalise within each broad_sector so scores reflect performance vs

sector peers

• Generate output/screener_output.xlsx — one sheet per preset with 20 KPI columns, sorted by composite score

descending

• Colour-code cells: green fill for cells meeting the preset threshold, red fill for cells failing the threshold



In [ ]:
# tests/screener/test_day17_18.py
import numpy as np
import pandas as pd
from src.screener.scoring import compute_composite_score
from src.screener.peer import compute_peer_percentiles


def test_composite_score_range_and_no_fabrication():
    df = pd.DataFrame({
        "company_id": [1, 2, 3, 4, 5],
        "return_on_equity_pct": [10, 15, 20, 25, 30],
        "net_profit_margin_pct": [5, 8, 10, 12, 15],
        "cash_from_operations_cr": [50, 60, 70, 80, 90],
        "net_profit": [40, 50, 60, 70, 80],
        "free_cash_flow_cr": [-5, 0, 10, 20, 30],
        "revenue_cagr_5yr": [5, 8, 10, 12, 15],
        "pat_cagr_5yr": [5, 8, 10, 12, 15],
        "debt_to_equity": [2, 1.5, 1, 0.5, 0],
        "ICR_effective": [2, 5, 10, np.inf, np.inf],
    })
    scored = compute_composite_score(df)
    assert scored["composite_quality_score"].between(0, 100).all()
    assert scored.loc[scored["company_id"] == 5, "composite_quality_score"].iloc[0] > \
           scored.loc[scored["company_id"] == 1, "composite_quality_score"].iloc[0]


def test_peer_percentile_no_group_handled_gracefully():
    universe = pd.DataFrame({
        "company_id": ["A", "B", "C"],
        "return_on_equity_pct": [10, 20, 30],
        "debt_to_equity": [1, 0.5, 2],
    })
    peer_groups = pd.DataFrame({"company_id": ["A", "B"], "peer_group_name": ["IT Services", "IT Services"]})
    result = compute_peer_percentiles(universe, peer_groups)
    c_rows = result[result["company_id"] == "C"]
    assert (c_rows["metric"] == "__no_group__").any()
    assert c_rows[c_rows["metric"] == "__no_group__"]["note"].iloc[0] == "No peer group assigned"


def test_de_inversion_in_peer_rank():
    universe = pd.DataFrame({
        "company_id": ["A", "B"],
        "debt_to_equity": [2.0, 0.5],  # B has lower D/E -> should get HIGHER percentile
    })
    peer_groups = pd.DataFrame({"company_id": ["A", "B"], "peer_group_name": ["X"]})
    result = compute_peer_percentiles(universe, peer_groups)
    de_rows = result[result["metric"] == "debt_to_equity"]
    b_rank = de_rows[de_rows["company_id"] == "B"]["percentile_rank"].iloc[0]
    a_rank = de_rows[de_rows["company_id"] == "A"]["percentile_rank"].iloc[0]
    assert b_rank > a_rank

Day 18 — Peer Percentile Rankings

• Write src/analytics/peer.py — load peer_groups.xlsx and compute PERCENT_RANK for 10 metrics within each

of 11 peer groups

• Metrics to rank: ROE, ROCE, Net Profit Margin, D/E (inverse — lower is better), FCF, PAT CAGR 5yr, Revenue

CAGR 5yr, EPS CAGR 5yr, Interest Coverage, Asset Turnover

• For D/E ranking: invert the percentile (1 - PERCENT_RANK) so that lower D/E = higher percentile rank

• Populate peer_percentiles table in SQLite with columns: company_id, peer_group_name, metric, value,

percentile_rank, year

• For companies not in any peer group: return message No peer group assigned — do not raise an error





In [ ]:
# scripts/run_day17_18.py
import sys
from pathlib import Path
project_root = Path(r"C:\Users\bhrra\Desktop\nifty100")
sys.path.insert(0, str(project_root))

import sqlite3
import pandas as pd
from src.screener.engine import build_screener_universe, run_all_presets, load_screener_config
from src.screener.export import export_screener_output
from src.screener.peer import compute_peer_percentiles, save_peer_percentiles_to_db

conn = sqlite3.connect(project_root / "db" / "nifty100.db")
universe = build_screener_universe(conn)
config = load_screener_config(project_root / "src"/"screener" / "screener_config.yaml")

# Day 17
results = run_all_presets(universe, conn, config)
export_screener_output(results, config, project_root / "output" / "screener_output.xlsx")
print("screener_output.xlsx written")
for name, df in results.items():
    print(f"{name}: top score = {df['composite_quality_score'].max()}")

# Day 18
peer_groups = pd.read_sql("SELECT company_id, peer_group_name FROM peer_groups", conn)
peer_pct = compute_peer_percentiles(universe, peer_groups)
save_peer_percentiles_to_db(peer_pct, conn, year=2024)  # confirm actual latest year
print("peer_percentiles rows written:", len(peer_pct))
print("Companies with no peer group:", (peer_pct["metric"] == "__no_group__").sum())

In [ ]:
for name, df in results.items():
    print(f"{name}: {len(df)} companies")

In [ ]:
for name, df in results.items():
    if len(df) > 0:
        print(f"\n{name} ({len(df)} companies):")
        print("  ROE nulls:", df["return_on_equity_pct"].isna().sum())
        print("  NPM nulls:", df["net_profit_margin_pct"].isna().sum())
        print("  CFO nulls:", df["cash_from_operations_cr"].isna().sum())
        print("  Net Profit nulls:", df["net_profit"].isna().sum())
        print("  Rev CAGR nulls:", df["revenue_cagr_5yr"].isna().sum())
        print("  PAT CAGR nulls:", df["pat_cagr_5yr"].isna().sum())
        print("  D/E nulls:", df["debt_to_equity"].isna().sum())
        print("  Has ICR_effective col:", "ICR_effective" in df.columns)

In [ ]:
for name, df in results.items():
    print(f"{name}: {len(df)} companies")

Day 19 — Radar Charts

• Generate radar/polar chart for each company in a peer group — 8 axes: ROE, ROCE, NPM, D/E, FCF score,

PAT CAGR 5yr, Revenue CAGR 5yr, Composite Score

• Each chart shows: the company's values as a filled polygon, peer group average as a dashed outline overlay

• Export as PNG to reports/radar_charts/ — filename format: _radar.png

• For companies with no peer group: generate a single-metric standalone chart with Nifty 100 average as reference

• Use matplotlib polar plot or plotly radar chart — ensure font size is readable at standard viewing size



Day 20 — Peer Comparison Excel Report

• Generate output/peer_comparison.xlsx with 11 sheets — one per peer group

• Each sheet columns: company_id, company_name, + 20 metric columns, + percentile rank for each metric

• Colour-code percentile rank cells: green fill for >= 75th percentile, yellow for 25th to 75th, red for <= 25th

percentile

• Highlight benchmark company row in each sheet with a gold/amber background

• Add a summary row at the bottom showing peer group median for each metric



In [ ]:
# tests/analytics/test_day19_20.py
import numpy as np
import pandas as pd
from pathlib import Path
import tempfile

from src.screener.radar import build_radar_scores, generate_radar_chart
from src.screener.peer_export import export_peer_comparison


def test_radar_scores_bounded_0_100():
    df = pd.DataFrame({
        "company_id": ["A", "B", "C", "D", "E"],
        "return_on_equity_pct": [10, 15, 20, 25, 30],
        "roce_percentage": [8, 12, 18, 22, 28],
        "net_profit_margin_pct": [5, 8, 10, 12, 15],
        "debt_to_equity": [2, 1.5, 1, 0.5, 0],
        "free_cash_flow_cr": [-10, 0, 50, 100, 200],
        "pat_cagr_5yr": [5, 8, 10, 12, 15],
        "revenue_cagr_5yr": [5, 8, 10, 12, 15],
        "composite_quality_score": [40, 50, 60, 70, 80],
    })
    scored = build_radar_scores(df)
    for col in [c for c in scored.columns if c.startswith("_radar_")]:
        valid = scored[col].dropna()
        assert (valid >= 0).all() and (valid <= 100).all()


def test_radar_handles_nan_composite_score_gracefully():
    """Composite score pending Day-17 fix -> should not crash chart generation."""
    df = pd.DataFrame({
        "company_id": ["A", "B"],
        "return_on_equity_pct": [10, 20], "roce_percentage": [10, 20],
        "net_profit_margin_pct": [5, 10], "debt_to_equity": [1, 0.5],
        "free_cash_flow_cr": [10, 20], "pat_cagr_5yr": [5, 10],
        "revenue_cagr_5yr": [5, 10],
        "composite_quality_score": [np.nan, np.nan],  # simulating the pending bug
    })
    scored = build_radar_scores(df)
    peer_groups = pd.DataFrame({"company_id": [], "peer_group_name": []})
    with tempfile.TemporaryDirectory() as tmp:
        path = generate_radar_chart("A", scored, peer_groups, tmp)
        assert path.exists()  # must not raise, even with NaN composite score


def test_peer_comparison_benchmark_highlight_and_median_row():
    peer_pct = pd.DataFrame([
        {"company_id": "TCS", "peer_group_name": "IT Services", "metric": "return_on_equity_pct", "value": 40, "percentile_rank": 0.9, "note": None},
        {"company_id": "INFY", "peer_group_name": "IT Services", "metric": "return_on_equity_pct", "value": 30, "percentile_rank": 0.4, "note": None},
    ])
    companies = pd.DataFrame({"id": ["TCS", "INFY"], "company_name": ["Tata Consultancy", "Infosys"]})
    peer_groups = pd.DataFrame({"company_id": ["TCS", "INFY"], "peer_group_name": ["IT Services", "IT Services"], "is_benchmark": [1, 0]})
    with tempfile.TemporaryDirectory() as tmp:
        out = export_peer_comparison(peer_pct, companies, peer_groups, Path(tmp) / "peer_comparison.xlsx")
        assert out.exists()
        sheet = pd.read_excel(out, sheet_name="IT Services")
        assert "MEDIAN" in sheet["company_id"].values

In [ ]:
# scripts/run_day19_20.py
import sys
from pathlib import Path
project_root = Path(r"C:\Users\bhrra\Desktop\nifty100")
sys.path.insert(0, str(project_root))
import importlib
import peer_export

importlib.reload(peer_export)
import sqlite3
import pandas as pd
from src.screener.engine import build_screener_universe, preprocess_special_metrics
from src.screener.scoring import compute_composite_score
from src.screener.radar import build_radar_scores, generate_all_radar_charts
from src.screener.peer import compute_peer_percentiles
from src.screener.peer_export import export_peer_comparison

conn = sqlite3.connect(project_root / "db" / "nifty100.db")
universe = build_screener_universe(conn)
universe = preprocess_special_metrics(universe)
universe = compute_composite_score(universe)   # Day 17 — currently NaN, will self-heal once fixed

scored = build_radar_scores(universe)
peer_groups = pd.read_sql("SELECT company_id, peer_group_name, is_benchmark FROM peer_groups", conn)
companies = pd.read_sql("SELECT id, company_name FROM companies", conn)

# Day 19
radar_paths = generate_all_radar_charts(scored, peer_groups, project_root / "reports" / "radar_charts")
print(f"Radar charts generated: {len(radar_paths)}")

# Day 20
peer_pct = compute_peer_percentiles(universe, peer_groups[["company_id", "peer_group_name"]])
out = export_peer_comparison(peer_pct, companies, peer_groups, project_root / "output" / "peer_comparison.xlsx")
print(f"peer_comparison.xlsx written: {out}")

Day 21 — Tests & Sprint Review

• Run all 14 DQ rule unit tests — must pass with 0 failures

• Verify screener: Quality Compounder preset returns companies with ROE > 15% and D/E < 1 — manually verify

top 5 results

• Verify peer rankings: within IT Services peer group, the company with highest ROE should have the highest ROE

percentile rank

• Sprint 3 retrospective

• Demo screener_output.xlsx and peer_comparison.xlsx to team lead

DELIVERABLES

• output/screener_output.xlsx — 6 sheets, one per preset, colour-coded cells

• output/peer_comparison.xlsx — 11 sheets, one per peer group, percentile colour-coded

• reports/radar_charts/ — PNG radar chart for each company with peer group overlay

• peer_percentiles table in SQLite — percentile ranks for all 11 groups

• config/screener_config.yaml — all threshold definitions, analyst-editable

• src/screener/engine.py — filter engine and composite score

• src/analytics/peer.py — peer percentile computation

EXIT CRITERIA (Definition of Done)

• 6 preset screeners each return between 5 and 50 companies

• peer_comparison.xlsx has exactly 11 sheets covering all 11 peer groups

• Peer percentile ranks are correct — verified by spot-checking IT Services and FMCG groups

• All 14 DQ rule unit tests pass

• Sprint 3 review meeting completed and signed off by team lead

In [ ]:
import sqlite3
conn = sqlite3.connect(r"C:\Users\bhrra\Desktop\nifty100\db\nifty100.db")  # adjust path if needed
cur = conn.cursor()
cur.execute("PRAGMA table_info(companies)")
for row in cur.fetchall():
    print(row)
conn.close()

In [ ]:
import sqlite3
conn = sqlite3.connect(r"C:\Users\bhrra\Desktop\nifty100\db\nifty100.db")  # adjust path if needed
cur = conn.cursor()
for t in ["companies","financial_ratios","cashflow","profitandloss","balancesheet",
          "analysis","prosandcons","peer_groups","sectors","documents"]:
    print(f"\n--- {t} ---")
    for row in cur.execute(f"PRAGMA table_info({t})"):
        print(row)
conn.close()

In [ ]:
import pandas as pd
df = pd.read_csv(r"C:\Users\bhrra\Desktop\nifty100\data\processed\market_cap.csv")   # adjust path
print(df.columns.tolist())
print(df.head(3))

import sqlite3
conn = sqlite3.connect(r"C:\Users\bhrra\Desktop\nifty100\db\nifty100.db")   # same path as before
cur = conn.cursor()

cur.execute("PRAGMA table_info(stock_prices)")
for row in cur.fetchall():
    print(row)

conn.close()

In [ ]:
import sqlite3
import pandas as pd
# conn = sqlite3.connect("nifty100.db")
conn = sqlite3.connect(r"C:\Users\bhrra\Desktop\nifty100\db\nifty100.db")   # same path as before

dupes = pd.read_sql_query(
    "SELECT company_id, COUNT(*) as cnt FROM sectors GROUP BY company_id ORDER BY cnt DESC LIMIT 10", conn
)
print(dupes)
conn.close()

In [5]:
import sqlite3
import pandas as pd
# conn = sqlite3.connect("nifty100.db")
conn = sqlite3.connect(r"C:\Users\bhrra\Desktop\nifty100\db\nifty100.db")   # same path as before

sample = pd.read_sql_query("SELECT * FROM companies;", conn)
print(sample)
conn.close()

             id                                       company_logo  \
0           ABB   https://mkt.in/static/mkt-icons/nifty100/ABB.png   
1    ADANIENSOL  https://m.economictimes.com/thumb/msid-1173715...   
2      ADANIENT  https://mkt.in/static/mkt-icons/nifty100/ADANI...   
3    ADANIGREEN  https://mkt.in/static/mkt-icons/nifty100/ADANI...   
4    ADANIPORTS  https://mkt.in/static/mkt-icons/nifty100/ADANI...   
..          ...                                                ...   
262         TCS      https://images.5paisa.com/MarketIcons/TCS.png   
263       TECHM  https://mkt.in/static/mkt-icons/nifty100/TECHM...   
264       TITAN  https://mkt.in/static/mkt-icons/nifty100/TITAN...   
265  TORNTPHARM  https://mkt.in/static/mkt-icons/nifty100/TORNT...   
266       TRENT  https://mkt.in/static/mkt-icons/nifty100/TRENT...   

                                  company_name  \
0                             Abbott India Ltd   
1                   Adani Energy Solutions Ltd   
2        

In [3]:
import sqlite3

conn = sqlite3.connect(r"C:\Users\bhrra\Desktop\nifty100\db\nifty100.db")

tables = conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table'"
).fetchall()

for table in tables:
    table_name = table[0]
    columns = conn.execute(
        f"PRAGMA table_info({table_name})"
    ).fetchall()

    print("\nTABLE:", table_name)
    print("COLUMNS:")
    for column in columns:
        print("  ", column[1])

conn.close()


TABLE: analysis
COLUMNS:
   id
   company_id
   compounded_sales_growth
   compounded_profit_growth
   stock_price_cagr
   roe

TABLE: balancesheet
COLUMNS:
   id
   company_id
   year
   equity_capital
   reserves
   borrowings
   other_liabilities
   total_liabilities
   fixed_assets
   cwip
   investments
   other_asset
   total_assets

TABLE: cashflow
COLUMNS:
   id
   company_id
   year
   operating_activity
   investing_activity
   financing_activity
   net_cash_flow

TABLE: companies
COLUMNS:
   id
   company_logo
   company_name
   chart_link
   about_company
   website
   nse_profile
   bse_profile
   face_value
   book_value
   roce_percentage
   roe_percentage

TABLE: documents
COLUMNS:
   id
   company_id
   Year
   Annual_Report

TABLE: financial_ratios
COLUMNS:
   id
   company_id
   year
   net_profit_margin_pct
   operating_profit_margin_pct
   return_on_equity_pct
   debt_to_equity
   interest_coverage
   asset_turnover
   free_cash_flow_cr
   capex_cr
   earnings_per

In [2]:
import sqlite3
conn = sqlite3.connect(r"C:\Users\bhrra\Desktop\nifty100\db\nifty100.db")
cur = conn.cursor()
cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
print(cur.fetchall())
conn.close()

[('analysis',), ('balancesheet',), ('cashflow',), ('companies',), ('documents',), ('financial_ratios',), ('market_cap',), ('peer_groups',), ('profitandloss',), ('prosandcons',), ('sectors',), ('stock_prices',), ('peer_percentiles',), ('valuation',)]


In [ ]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(r"C:\Users\bhrra\Desktop\nifty100\db\nifty100.db")
patterns = pd.read_sql_query("SELECT company_id, capital_allocation_label FROM financial_ratios", conn)
companies = pd.read_sql_query("SELECT id AS company_id, company_name FROM companies", conn)
merged = patterns.merge(companies, on="company_id", how="left")
print(merged[merged["company_name"].isna()])
conn.close()

In [ ]:
df.isna().sum()

In [1]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(r"C:\Users\bhrra\Desktop\nifty100\db\nifty100.db")

# Does the column exist at all?
cur = conn.cursor()
cur.execute("PRAGMA table_info(financial_ratios)")
cols = [row[1] for row in cur.fetchall()]
print("capital_allocation_label in columns?", "capital_allocation_label" in cols)

# If it exists, how many non-null values does it actually have?
if "capital_allocation_label" in cols:
    df = pd.read_sql_query("SELECT capital_allocation_label FROM financial_ratios", conn)
    print(df["capital_allocation_label"].value_counts(dropna=False))

conn.close()

capital_allocation_label in columns? True
capital_allocation_label
None                     3117
Reinvestor                555
Mixed                     185
Growth Funded by Debt      92
Liquidating Assets         90
Distress Signal            36
Unclassified               15
Pre-Revenue                 8
Cash Accumulator            2
Name: count, dtype: int64


In [ ]:
a = [1,2,2,3]
if all(a.count(i) == 1 for  i in a):
    print( -1) 
             